# Handling parameters in ParaQeet

The `OptimizationMap` is a utility class that collects all parameters
that shall be considered during optimization and associates them with
the corresponding `Optimizable` interface.
With this class, `Quantity` objects can be traced back to the `Optimizable` to
which they belong.
Before optimization, an instance of this class needs to be filled and
passed to the optimizer.

In [ ]:
from paraqeet.optimization_map import OptimizationMap

from paraqeet.signal.iq_mixer import IQMixer
from paraqeet.signal.envelopes import ConstantEnvelope, FlatTopGaussianEnvelope
from paraqeet.quantity import Quantity
from paraqeet.optimizable import Optimizable

## Example devices and their parameters
We define a tone and a signal generator and look at their parameters:

In [ ]:
tone = ConstantEnvelope()

gen = IQMixer(envelopes=[tone])

We note that both `tone` and `gen` are instances of `Optimizable`.

In [ ]:
print(isinstance(tone, Optimizable))
print(isinstance(gen, Optimizable))

As such they must have a `get_parameters` method implemented. We can obtain the (default) parameters of the tone as

In [ ]:
params_tone = tone.get_parameters()
print(params_tone)

and those of the generator as

In [ ]:
params_gen = gen.get_parameters()
print(params_gen)

As we see, in this case, `gen` has the parameters of the `tone` as its parameters. Moreover, it has two additional  parameters, namely, the frequency of the local oscillator ('lo_freq') and the phase of the pulse ('Phase'). We also note that the `get_parameters` returns a list of ParaQeet's `Quantity` objects: 

In [ ]:
print(isinstance(params_gen, list))
print([isinstance(param, Quantity) for param in params_gen])

## The Optimization Map
To handle the parameters of both tones, we make an `OptimizationMap`and add the parameters of the `gen` explcitely.

In [ ]:
optmap = OptimizationMap()
optmap.add(gen, params_gen)

Note that we also need to pass `gen` itself when we add the corresponding parameters to the optmap. We can get a list output of all parameters with

In [ ]:
optmap.get_all_parameters()

If we just want to optimize the frequency, we set

In [ ]:
optmap.add(gen, [params_gen[2]])
optmap.get_all_parameters()

which overwrites the previous parameters associated with `gen`.

## Adding more devices
As a second drive, we create a signal shaped by a flat-top Gaussian envelope:

In [ ]:
tone_2 = FlatTopGaussianEnvelope()
gen_2 = IQMixer(envelopes=[tone_2])

If we don't specify an explicit list of parameters, all of them get added.

In [ ]:
optmap.add(gen_2)
optmap.get_all_parameters()

If we just print `optmap` we can also see that the parameters are associated with different objects

In [ ]:
print(optmap)

## Selecting parameters
There is a convenient filter method to select
parameters based on properties.
The following example selects all amplitudes:

In [ ]:
def all_amplitudes(par):
    """Get the amplitudes."""
    return par.get_name() == "Amplitude"


optmap.filter_parameters(all_amplitudes)
optmap.get_all_parameters()

In [ ]:
print(optmap)

Adding back all parameters:

In [ ]:
optmap.add(gen)
optmap.add(gen_2)
print(optmap)

Now, we select every parameter with unit "Hz":

In [ ]:
def hz_filter(par):
    """Get every parameter with the unit 'Hz'."""
    return par.get_unit() == "Hz"


optmap.filter_parameters(hz_filter)
print(optmap)

In the following notebooks we will put to practice what we learned in this notebook and use the `OptimizationMap` to select the parameters to optimize in optimal control tasks. 